In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import pandas as pd
import numpy as np

# ============================================
# 1. Custom Dataset for DataFrame-based inputs
# ============================================

class ImageDFDataset(Dataset):
    def __init__(self, df, label_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = '../data' + self.df.loc[idx, "image_path"]
        label_str = self.df.loc[idx, "label"]
        label = self.label_to_idx[label_str]

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


# ============================================
# 2. Build label mapping
# ============================================

def build_label_mapping(df_train):
    classes = sorted(df_train["label"].unique())
    label_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_label = {i: c for c, i in label_to_idx.items()}
    return label_to_idx, idx_to_label


# ============================================
# 3. Transforms
# ============================================

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(144),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# ============================================
# 4. Create datasets & loaders from dataframe
# ============================================

def make_dataloaders(df_train, df_val, batch_size=32):

    label_to_idx, idx_to_label = build_label_mapping(df_train)
    num_classes = len(label_to_idx)

    train_dataset = ImageDFDataset(df_train, label_to_idx, transform=train_transform)
    val_dataset   = ImageDFDataset(df_val, label_to_idx, transform=val_transform)

    # ---- Balanced sampler (important for ~20 images/class) ----
    class_counts = df_train["label"].value_counts().sort_index()
    class_weights = 1.0 / torch.tensor(class_counts.tolist(), dtype=torch.float)

    sample_weights = [
        class_weights[label_to_idx[label]]
        for label in df_train["label"]
    ]
    print(len(sample_weights))

    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, num_classes, label_to_idx, idx_to_label


# ============================================
# 5. Build ResNet-18 (train from scratch)
# ============================================

def build_resnet18(num_classes):
    model = models.resnet18(weights=None)   # NOT pretrained

    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(512, num_classes)
    )

    return model


# ============================================
# 6. Training loop (clean version)
# ============================================

def train_model(model, train_loader, val_loader, epochs=100, lr=1e-3):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        running_loss = 0


        for imgs, labels in train_loader:
            optimizer.zero_grad()
            imgs, labels_a, labels_b, lam = mixup(imgs, labels)
            imgs, labels_a, labels_b  = imgs.to(device), labels_a.to(device), labels_b.to(device)
            outputs = model(imgs)
            loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch {epoch+1}/{epochs} "
              f"| Train Loss: {running_loss/len(train_loader):.4f} "
              f"| Val Loss: {val_loss:.4f}")


# ============================================
# 7. Validation
# ============================================

@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()

    return total_loss / len(loader)


def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam





In [7]:
from sklearn.model_selection import train_test_split
# ============================================
# 8. Example usage
# ============================================
df = pd.read_csv("../data/train_images.csv")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_loader, val_loader, num_classes, label_to_idx, idx_to_label = \
    make_dataloaders(train_df, val_df)

model = build_resnet18(200)

3140


In [8]:
train_model(model, train_loader, val_loader, epochs=120, lr=1e-3)

Epoch 1/120 | Train Loss: 5.5643 | Val Loss: 5.3183
Epoch 2/120 | Train Loss: 5.3542 | Val Loss: 5.2614
Epoch 3/120 | Train Loss: 5.3317 | Val Loss: 5.2838
Epoch 4/120 | Train Loss: 5.2636 | Val Loss: 5.3084
Epoch 5/120 | Train Loss: 5.2156 | Val Loss: 5.1353
Epoch 6/120 | Train Loss: 5.1858 | Val Loss: 5.0949
Epoch 7/120 | Train Loss: 5.1650 | Val Loss: 5.1097
Epoch 8/120 | Train Loss: 5.1532 | Val Loss: 4.9937
Epoch 9/120 | Train Loss: 5.1375 | Val Loss: 4.9614
Epoch 10/120 | Train Loss: 5.0847 | Val Loss: 5.0209
Epoch 11/120 | Train Loss: 5.0913 | Val Loss: 4.9383
Epoch 12/120 | Train Loss: 5.0214 | Val Loss: 5.0194
Epoch 13/120 | Train Loss: 5.0103 | Val Loss: 4.8324
Epoch 14/120 | Train Loss: 4.9471 | Val Loss: 4.9068
Epoch 15/120 | Train Loss: 4.9750 | Val Loss: 4.8149
Epoch 16/120 | Train Loss: 4.9756 | Val Loss: 4.8350
Epoch 17/120 | Train Loss: 4.8892 | Val Loss: 4.7614
Epoch 18/120 | Train Loss: 4.8735 | Val Loss: 4.7375
Epoch 19/120 | Train Loss: 4.8405 | Val Loss: 4.6713
Ep

KeyboardInterrupt: 

In [9]:
test_df = pd.read_csv("../data/test_images_path.csv")
test_dataset   = ImageDFDataset(test_df, label_to_idx, transform=val_transform)
test_loader   = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [10]:
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [11]:
all_ids = test_df["id"].tolist()
all_preds = []

In [12]:
with torch.no_grad():
    for inputs, _ in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.numpy())

In [13]:
predicted_labels = [idx_to_label[i] for i in all_preds]

In [14]:
output_df = pd.DataFrame({
    "id": all_ids,
    "label": predicted_labels
})

output_df.to_csv("test_predictions.csv", index=False)
print("Saved test_predictions.csv!")

Saved test_predictions.csv!


In [15]:
from sklearn.metrics import accuracy_score
val_preds = []
val_labels = []
model.eval()

with torch.no_grad():
    for inputs, labels in val_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        val_preds.extend(predicted.numpy())
        val_labels.extend(labels.numpy())

accuracy = accuracy_score(val_labels, val_preds)

In [16]:
accuracy

0.28880407124681934